In [0]:
# ============================================================
# DATABRICKS PYSPARK EDA NOTEBOOK CELL
# Dataset: structured_traffic.csv
# ============================================================

# -----------------------------
# 1. Import libraries
# -----------------------------
from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# 2. Load the CSV file
# -----------------------------
# Replace this path if your Databricks upload path is different
file_path = "/Volumes/workspace/default/iot_class_gw/structured_traffic.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(file_path)
)

# -----------------------------
# 3. Basic dataset structure
# -----------------------------
print("========== FIRST 5 ROWS ==========")
display(df.limit(5))

print("========== SCHEMA ==========")
df.printSchema()

print("========== NUMBER OF OBSERVATIONS ==========")
num_rows = df.count()
num_cols = len(df.columns)
print(f"Number of rows: {num_rows}")
print(f"Number of columns: {num_cols}")

print("========== COLUMN NAMES ==========")
print(df.columns)

# -----------------------------
# 4. Make sure date/timestamp are in the correct format
# -----------------------------
df = (
    df.withColumn("date_parsed", F.to_date("date", "yyyy-MM-dd"))
      .withColumn("timestamp_parsed", F.to_timestamp("timestamp", "yyyy-MM-dd HH:mm:ss"))
)

print("========== DATE CHECK ==========")
display(df.select("date", "date_parsed", "timestamp", "timestamp_parsed").limit(5))

# -----------------------------
# 5. Summary statistics table
# -----------------------------
print("========== SUMMARY STATISTICS ==========")
summary_stats = df.describe()
display(summary_stats)

print("========== SUMMARY STATISTICS FOR TRAFFIC VOLUME ==========")
traffic_summary = df.select("traffic_volume").describe()
display(traffic_summary)

# -----------------------------
# 6. Histogram of traffic volume
# -----------------------------
traffic_pd = df.select("traffic_volume").dropna().toPandas()

plt.figure(figsize=(10, 6))
plt.hist(traffic_pd["traffic_volume"], bins=30, edgecolor="black")
plt.title("Histogram of Traffic Volume")
plt.xlabel("Traffic Volume")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# -----------------------------
# 7. Box plot of traffic volume
# -----------------------------
plt.figure(figsize=(8, 6))
plt.boxplot(traffic_pd["traffic_volume"].dropna())
plt.title("Box Plot of Traffic Volume")
plt.ylabel("Traffic Volume")
plt.tight_layout()
plt.show()

# -----------------------------
# 8. Line chart of average traffic by hour of day
# -----------------------------
avg_by_hour = (
    df.groupBy("hour")
      .agg(F.avg("traffic_volume").alias("avg_traffic_volume"))
      .orderBy("hour")
)

print("========== AVERAGE TRAFFIC BY HOUR ==========")
display(avg_by_hour)

avg_by_hour_pd = avg_by_hour.toPandas()

plt.figure(figsize=(10, 6))
plt.plot(avg_by_hour_pd["hour"], avg_by_hour_pd["avg_traffic_volume"], marker="o")
plt.title("Average Traffic Volume by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Average Traffic Volume")
plt.xticks(range(0, 24))
plt.grid(True)
plt.tight_layout()
plt.show()

# -----------------------------
# 9. Bar chart of average traffic by day of week
# -----------------------------
# Order days manually so the chart appears in calendar order
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

avg_by_day = (
    df.groupBy("day_of_week")
      .agg(F.avg("traffic_volume").alias("avg_traffic_volume"))
)

avg_by_day_pd = avg_by_day.toPandas()

# Convert day_of_week to ordered categorical for correct sorting
avg_by_day_pd["day_of_week"] = pd.Categorical(
    avg_by_day_pd["day_of_week"],
    categories=day_order,
    ordered=True
)
avg_by_day_pd = avg_by_day_pd.sort_values("day_of_week")

print("========== AVERAGE TRAFFIC BY DAY OF WEEK ==========")
display(spark.createDataFrame(avg_by_day_pd))

plt.figure(figsize=(10, 6))
plt.bar(avg_by_day_pd["day_of_week"], avg_by_day_pd["avg_traffic_volume"])
plt.title("Average Traffic Volume by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Average Traffic Volume")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# -----------------------------
# 10. Bar chart of top high-volume locations
# -----------------------------
# Here, a "location" is defined using roadway + direction + from/to + segment_id
top_locations = (
    df.groupBy("segment_id", "roadway__name", "from", "to", "direction")
      .agg(F.avg("traffic_volume").alias("avg_traffic_volume"))
      .orderBy(F.desc("avg_traffic_volume"))
      .limit(10)
)

print("========== TOP 10 HIGH-VOLUME LOCATIONS ==========")
display(top_locations)

top_locations_pd = top_locations.toPandas()

top_locations_pd["location_label"] = (
    top_locations_pd["roadway__name"].astype(str) + " | " +
    top_locations_pd["direction"].astype(str) + " | Seg " +
    top_locations_pd["segment_id"].astype(str)
)

plt.figure(figsize=(12, 7))
plt.bar(top_locations_pd["location_label"], top_locations_pd["avg_traffic_volume"])
plt.title("Top 10 High-Volume Locations")
plt.xlabel("Location")
plt.ylabel("Average Traffic Volume")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# -----------------------------
# 11. Heat map of average traffic by day and hour
# -----------------------------
heatmap_data = (
    df.groupBy("day_of_week", "hour")
      .agg(F.avg("traffic_volume").alias("avg_traffic_volume"))
      .toPandas()
)

# Put days in the correct order
heatmap_data["day_of_week"] = pd.Categorical(
    heatmap_data["day_of_week"],
    categories=day_order,
    ordered=True
)

heatmap_data = heatmap_data.sort_values(["day_of_week", "hour"])

# Create pivot table: rows = day of week, columns = hour
heatmap_pivot = heatmap_data.pivot(
    index="day_of_week",
    columns="hour",
    values="avg_traffic_volume"
)

print("========== HEAT MAP TABLE ==========")
display(heatmap_pivot)

plt.figure(figsize=(14, 6))
plt.imshow(heatmap_pivot, aspect="auto")
plt.colorbar(label="Average Traffic Volume")
plt.title("Heat Map of Average Traffic Volume by Day and Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Day of Week")
plt.xticks(range(len(heatmap_pivot.columns)), heatmap_pivot.columns)
plt.yticks(range(len(heatmap_pivot.index)), heatmap_pivot.index)
plt.tight_layout()
plt.show()

# -----------------------------
# 12. Optional: Missing values check
# -----------------------------
print("========== MISSING VALUES CHECK ==========")
missing_values = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])
display(missing_values)

print("========== EDA COMPLETE ==========")